# Load libraries

In [ ]:
import os
import gc
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import seaborn as sns

import skopt
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.preprocessing import minmax_scale

from sklearn.neighbors import NearestNeighbors,KNeighborsClassifier, NeighborhoodComponentsAnalysis, KNeighborsRegressor, LocalOutlierFactor

import xgboost as xgb
from xgboost import XGBRegressor
import lightgbm as lgb

import optiver_training_dataset_build as opt_train

# Custom functions

In [ ]:
def pickle_dump(path, saveobj):
    import pickle
    filehandler = open(path,"wb")
    pickle.dump(saveobj,filehandler)
    print("File pickled")
    filehandler.close()

In [ ]:
def pickle_load(path):
    import pickle
    file = open(path,'rb')
    loadobj = pickle.load(file)
    file.close()
    return loadobj

In [ ]:
# Function to early stop with root mean squared percentage error
def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(np.square((y_true - y_pred) / y_true)))


In [ ]:
def feval_rmspe(y_pred, lgb_train):
    y_true = lgb_train.get_label()
    return 'RMSPE', rmspe(y_true, y_pred), False

# Read in data

In [ ]:
train_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/train.csv")
test_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/test.csv")
submit_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/sample_submission.csv")

In [ ]:
display(train_df.head(2))
display(test_df.head(2))
display(submit_df.head(2))

# Build training datasets

## Illidan

In [ ]:
opt_illi_df = opt_train.build_dataset()

print(opt_illi_df.shape)
opt_illi_df.head()

In [ ]:
illiCols = pickle_load("/kaggle/input/optiver-training-data/illiCols.pkl")

opt_illi_df = opt_illi_df[['stock_id','time_id','target'] + illiCols]

opt_illi_df.shape

## Public data small

In [ ]:
public1_df = pd.read_feather("/kaggle/input/optiver-training-data/publicFeats1.feather")

print(public1_df.shape)
public1_df.head()

In [ ]:
opt_illi_df = pd.merge(opt_illi_df, public1_df, on=['time_id','stock_id','target'], how="left")
opt_illi_df.shape

In [ ]:
del public1_df
_ = gc.collect()

## Public data big

In [ ]:
public2_df = pd.read_feather("/kaggle/input/optiver-training-data/publicFeats2.feather")

print(public2_df.shape)
public2_df.head()

In [ ]:
public1Cols = pickle_load("/kaggle/input/optiver-training-data/public1Cols.pkl")

public2_df = public2_df[['stock_id','time_id','target'] + public1Cols]

public2_df.shape

In [ ]:
opt_illi_df = pd.merge(opt_illi_df, public2_df, on=['time_id','stock_id','target'], how="left")
opt_illi_df.shape

In [ ]:
del public2_df
_ = gc.collect()

# Correlation KO

In [ ]:
def correlation_KO(dataset, threshold):
    col_corr = set()  # Set of all the names of correlated columns
    corr_matrix = dataset.corr()
    for i in tqdm(range(len(corr_matrix.columns))):
        for j in range(i):
            if abs(corr_matrix.iloc[i, j]) >= threshold: # we are interested in absolute coeff value
                colname = corr_matrix.columns[i]  # getting the name of column
                col_corr.add(colname)
    return col_corr

In [ ]:
corr_cols = correlation_KO(opt_illi_df, 0.999)
len(set(corr_cols))

In [ ]:
corr_cols

In [ ]:
opt_illi_df.drop(corr_cols,axis=1,inplace=True)

print(opt_illi_df.shape)

# Model dataset

In [ ]:
opt_train_df = opt_illi_df.copy()
opt_train_df.shape

# Data split

In [ ]:
# opt_train_df['fold'] = -1

# group_kfold = GroupKFold(n_splits=5)

# kfold_val_dict = dict()

# for fold, (_,val_idx) in enumerate(group_kfold.split(opt_train_df,opt_train_df['target'].values,opt_train_df['time_id'])):
#     opt_train_df.loc[val_idx, 'fold'] = fold
    
#     kfold_val_dict[fold] = val_idx


# print(opt_train_df.shape)

# opt_train_df['fold'].value_counts()

In [ ]:
opt_train_df['fold'] = -1

# Create a KFold object
kfold = KFold(n_splits = 5, random_state = 2021, shuffle = True)

kfold_val_dict = dict()

for fold, (_,val_idx) in enumerate(kfold.split(opt_train_df)):
    opt_train_df.loc[val_idx, 'fold'] = fold
    
    kfold_val_dict[fold] = val_idx


print(opt_train_df.shape)

opt_train_df['fold'].value_counts()

In [ ]:
opt_illi_df.reset_index(drop=True)
opt_illi_df.to_feather("./IlliModel_df.feather")

# KFold training 1

In [ ]:
seed0=2021
params0 = {
    'objective': 'rmse',
    'boosting_type': 'gbdt',
    'max_depth': 5,
    'max_bin':100,
    'min_data_in_leaf':500,
    'learning_rate': 0.05,
    'subsample': 0.7,
    'subsample_freq': 4,
    'feature_fraction': 0.5,
    'lambda_l1': 0.5,
    'lambda_l2': 10,
    'categorical_column':[0],
    'seed':seed0,
    'feature_fraction_seed': seed0,
    'bagging_seed': seed0,
    'drop_seed': seed0,
    'data_random_seed': seed0,
    'n_jobs':-1,
    'verbose': -1}
seed1=42
params1 = {
        'learning_rate': 0.1,        
        'lambda_l1': 2,
        'lambda_l2': 7,
        'num_leaves': 800,
        'min_sum_hessian_in_leaf': 20,
        'feature_fraction': 0.8,
        'feature_fraction_bynode': 0.8,
        'bagging_fraction': 0.9,
        'bagging_freq': 42,
        'min_data_in_leaf': 700,
        'max_depth': 4,
        'categorical_column':[0],
        'seed': seed1,
        'feature_fraction_seed': seed1,
        'bagging_seed': seed1,
        'drop_seed': seed1,
        'data_random_seed': seed1,
        'objective': 'rmse',
        'boosting': 'gbdt',
        'verbosity': -1,
        'n_jobs':-1,
    }

In [ ]:
# StratifiedGroupKFold - Run training
oof_preds = np.zeros(opt_train_df.shape[0])

var_imp = pd.DataFrame()

params = params0

for fold in tqdm(range(5)):

    print(f"Training fold {fold+1}")
    
    trn_x = opt_train_df[opt_train_df['fold']!=fold].drop(columns=['time_id','target','fold','row_id'])
    trn_y = opt_train_df[opt_train_df['fold']!=fold]['target'].values
    
    val_x =opt_train_df[opt_train_df['fold']==fold].drop(columns=['time_id','target','fold','row_id'])
    val_y = opt_train_df[opt_train_df['fold']==fold]['target'].values
    
    val_idx = kfold_val_dict[fold]
     
    # Root mean squared percentage error weights
    train_weights = 1 / np.square(trn_y)
    val_weights = 1 / np.square(val_y)
    train_dataset = lgb.Dataset(trn_x, trn_y, weight = train_weights)
    val_dataset = lgb.Dataset(val_x, val_y, weight = val_weights)
    model = lgb.train(params = params,
                      num_boost_round=10000,
                      train_set = train_dataset, 
                      valid_sets = [train_dataset, val_dataset], 
                      verbose_eval = 50,
                      early_stopping_rounds=50,
                      feval = feval_rmspe)
    
    
    var_imp[f'Fold{fold+1}'] = pd.Series(model.feature_importance(importance_type='gain'), index=trn_x.columns)
    
    oof_preds[val_idx] = model.predict(val_x)
    
    pickle_dump(f"./lgbm_illi_public_261_fold{fold+1}.pkl", model)
    
    R2 = round(r2_score(y_true = val_y, y_pred = oof_preds[val_idx]),3)
    RMSPE = round(rmspe(y_true = val_y, y_pred = oof_preds[val_idx]),3)
    print(f'Fold {fold+1}: R2 score: {R2}, RMSPE: {RMSPE}')
    
    del model, trn_x, trn_y, val_x, val_y
    _ = gc.collect()

R2 = round(r2_score(y_true = opt_train_df['target'].values, y_pred = oof_preds),3)
RMSPE = round(rmspe(y_true = opt_train_df['target'].values, y_pred = oof_preds),3)
print(f'OOF: R2 score: {R2}, RMSPE: {RMSPE}')

In [ ]:
print(oof_preds.shape)

pickle_dump("./oof_lgb_illi.pkl", oof_preds)

In [ ]:
!ls -hlt

# Feature importance - KFold 1

In [ ]:
varImp_df = pd.DataFrame(var_imp)
varImp_df['Avg'] = (varImp_df['Fold1'] + varImp_df['Fold2'] + varImp_df['Fold3'] + varImp_df['Fold4'] + varImp_df['Fold5']) / 5
varImp_df.shape

In [ ]:
plt.hist(var_imp['Avg'].sort_values(ascending=False))

In [ ]:
plt.hist(var_imp['Avg'].sort_values(ascending=False)[var_imp['Avg'].sort_values(ascending=False)<1000])

In [ ]:
_ = plt.hist(var_imp['Avg'].sort_values(ascending=False)[var_imp['Avg'].sort_values(ascending=False)<200], bins=100)

In [ ]:
varImp_df['Avg'].nlargest(30).plot(kind='barh', figsize=(20,10))
plt.show()

In [ ]:
temp = pd.DataFrame(varImp_df['Avg'].sort_values(ascending=False)).reset_index()

temp.columns = ['vars','imp']

# temp[temp['vars']=='noise']

In [ ]:
modelCols = varImp_df['Avg'].sort_values(ascending=False)[varImp_df['Avg'].sort_values(ascending=False) > 200].index.tolist()
len(modelCols)

In [ ]:
modelCols

# KFold training 2

In [ ]:
seed0=2021
params = {
    'objective': 'rmse',
    'boosting_type': 'gbdt',
    'max_depth': -1,
    'max_bin':100,
    'min_data_in_leaf':500,
    'learning_rate': 0.05,
    'subsample': 0.72,
    'subsample_freq': 4,
    'feature_fraction': 0.5,
    'lambda_l1': 0.5,
    'lambda_l2': 1.0,
#     'categorical_column':[ind for ind,col in enumerate(opt_train_df.drop(columns=['time_id','target','fold','row_id']).columns) if col in ['stock_id','eqtyp_cluster','kmn7_cluster','hbd_cluster']],
    'seed':seed0,
    'feature_fraction_seed': seed0,
    'bagging_seed': seed0,
    'drop_seed': seed0,
    'data_random_seed': seed0,
    'n_jobs':-1,
    'device':'gpu',
    'verbose': -1}

In [ ]:
seed0=2021
params0 = {
    'objective': 'rmse',
    'boosting_type': 'gbdt',
    'max_depth': -1,
    'max_bin':100,
    'min_data_in_leaf':500,
    'learning_rate': 0.05,
    'subsample': 0.72,
    'subsample_freq': 4,
    'feature_fraction': 0.5,
    'lambda_l1': 0.5,
    'lambda_l2': 1.0,
    'categorical_column':[0],
    'seed':seed0,
    'feature_fraction_seed': seed0,
    'bagging_seed': seed0,
    'drop_seed': seed0,
    'data_random_seed': seed0,
    'n_jobs':-1,
    'verbose': -1}

In [ ]:
# StratifiedGroupKFold - Run training
oof_preds = np.zeros(opt_train_df.shape[0])

var_imp = pd.DataFrame()

params = params0

for fold in tqdm(range(5)):

    print(f"Training fold {fold+1}")
    
    trn_x = opt_train_df[opt_train_df['fold']!=fold].drop(columns=['time_id','target','fold'])[modelCols]
    trn_y = opt_train_df[opt_train_df['fold']!=fold]['target'].values
    
    val_x =opt_train_df[opt_train_df['fold']==fold].drop(columns=['time_id','target','fold'])[modelCols]
    val_y = opt_train_df[opt_train_df['fold']==fold]['target'].values
    
    val_idx = kfold_val_dict[fold]
     
    # Root mean squared percentage error weights
    train_weights = 1 / np.square(trn_y)
    val_weights = 1 / np.square(val_y)
    train_dataset = lgb.Dataset(trn_x, trn_y, weight = train_weights)
    val_dataset = lgb.Dataset(val_x, val_y, weight = val_weights)
    model = lgb.train(params = params,
                      num_boost_round=10000,
                      train_set = train_dataset, 
                      valid_sets = [train_dataset, val_dataset], 
                      verbose_eval = 50,
                      early_stopping_rounds=50,
                      feval = feval_rmspe)
    
    
    var_imp[f'Fold{fold+1}'] = pd.Series(model.feature_importance(importance_type='gain'), index=trn_x.columns)
    
    oof_preds[val_idx] = model.predict(val_x)
    
    R2 = round(r2_score(y_true = val_y, y_pred = oof_preds[val_idx]),3)
    RMSPE = round(rmspe(y_true = val_y, y_pred = oof_preds[val_idx]),3)
    print(f'Fold {fold+1}: R2 score: {R2}, RMSPE: {RMSPE}')
    
    del model, trn_x, trn_y, val_x, val_y
    _ = gc.collect()

R2 = round(r2_score(y_true = opt_train_df['target'].values, y_pred = oof_preds),3)
RMSPE = round(rmspe(y_true = opt_train_df['target'].values, y_pred = oof_preds),3)
print(f'OOF: R2 score: {R2}, RMSPE: {RMSPE}')

# Feature importance - KFold2

In [ ]:
varImp_df = pd.DataFrame(var_imp)
varImp_df['Avg'] = (varImp_df['Fold1'] + varImp_df['Fold2'] + varImp_df['Fold3'] + varImp_df['Fold4'] + varImp_df['Fold5']) / 5
varImp_df.shape

In [ ]:
plt.hist(var_imp['Avg'].sort_values(ascending=False))

In [ ]:
plt.hist(var_imp['Avg'].sort_values(ascending=False)[var_imp['Avg'].sort_values(ascending=False)<1000])

In [ ]:
varImp_df['Avg'].nlargest(30).plot(kind='barh', figsize=(20,10))
plt.show()

In [ ]:
modelCols = varImp_df['Avg'].sort_values(ascending=False)[varImp_df['Avg'].sort_values(ascending=False) > 200].index.tolist()
len(modelCols)

# Skopt training

In [ ]:
seed0=2021
SEARCH_PARAMS = {'learning_rate': 0.4,
                'max_depth': 4,
                'num_leaves': 32,
                'feature_fraction': 0.8,
                'subsample': 0.2,
                'lambda_l1': 0.5,
                'lambda_l2': 1.0,}



FIXED_PARAMS= {
                'objective': 'rmse',
                'boosting':'gbdt',
                'num_boost_round':10000,
                'early_stopping_rounds':50, 
                'min_data_in_leaf':500,
                'max_bin':300,
                'seed':seed0,
                'feature_fraction_seed': seed0,
                'bagging_seed': seed0,
                'drop_seed': seed0,
                'data_random_seed': seed0,
                'n_jobs':-1,
#                 'device':'gpu',
                'verbose': -1}

In [ ]:
def train_evaluate(search_params):
    
    fold=0
    
    X_train = opt_train_df[opt_train_df['fold']!=fold].drop(columns=['time_id','target','fold','row_id'])
    y_train = opt_train_df[opt_train_df['fold']!=fold]['target'].values
    
    X_valid =opt_train_df[opt_train_df['fold']==fold].drop(columns=['time_id','target','fold','row_id'])
    y_valid = opt_train_df[opt_train_df['fold']==fold]['target'].values
    
    # train_data = lgb.Dataset(X_train, label=y_train)
    # valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)

    # Root mean squared percentage error weights
    train_weights = 1 / np.square(y_train)
    val_weights = 1 / np.square(y_valid)
    train_dataset = lgb.Dataset(X_train, y_train, weight = train_weights)
    val_dataset = lgb.Dataset(X_valid, y_valid, weight = val_weights)

    params = {
             'objective':FIXED_PARAMS['objective'],
             'min_data_in_leaf':FIXED_PARAMS['min_data_in_leaf'],
#              'device':FIXED_PARAMS['device'],
             'boosting':FIXED_PARAMS['boosting'],
             'n_jobs':FIXED_PARAMS['n_jobs'],
             'verbose':FIXED_PARAMS['verbose'],
             **search_params}
    
    evals_result = {}

    model = lgb.train(params = params,
                      num_boost_round=FIXED_PARAMS['num_boost_round'],
                      train_set = train_dataset, 
                      valid_sets = [train_dataset, val_dataset], 
                      verbose_eval = 50, 
                      early_stopping_rounds=FIXED_PARAMS['early_stopping_rounds'],
                      valid_names=['train','valid'],
                      evals_result=evals_result,
                      feval = feval_rmspe)

    score = model.best_score['valid']['RMSPE']
    return score

In [ ]:
SPACE = [
    skopt.space.Real(0.01, 0.5, name='learning_rate', prior='log-uniform'),
    skopt.space.Integer(1, 10, name='max_depth'),
#     skopt.space.Integer(500, 5000, name='min_data_in_leaf'),
    skopt.space.Integer(30, 300, name='num_leaves'),
    skopt.space.Real(0.5, 1.0, name='feature_fraction', prior='uniform'),
    skopt.space.Real(0.4, 1.0, name='subsample', prior='uniform'),
    skopt.space.Real(0.001, 1000, name='lambda_l1', prior='log-uniform'),
    skopt.space.Real(0.001, 1000, name='lambda_l2', prior='log-uniform'),
]
@skopt.utils.use_named_args(SPACE)

def objective(**params):
    return train_evaluate(params)


results = skopt.forest_minimize(objective, SPACE, 
                                n_calls=5, n_initial_points=5, random_state = 17,
                                verbose=True
#                                 callback=[monitor]
                               )

In [ ]:
skopt_df = pd.DataFrame()

learning_rate = []
max_depth = []
num_leaves = []
feature_fraction = []
subsample = []
lambda_l1 = []
lambda_l2 = []
RMSPE = []

for params, score in zip(results.x_iters, results.func_vals):
    learning_rate.append(params[0])
    max_depth.append(params[1])
    num_leaves.append(params[2])
    feature_fraction.append(params[3])
    subsample.append(params[4])
    lambda_l1.append(params[5])
    lambda_l2.append(params[6])
    RMSPE.append(score)

skopt_df['learning_rate'] = learning_rate 
skopt_df['max_depth'] = max_depth 
skopt_df['num_leaves'] = num_leaves 
skopt_df['feature_fraction'] = feature_fraction 
skopt_df['subsample'] = subsample 
skopt_df['lambda_l1'] = lambda_l1 
skopt_df['lambda_l2'] = lambda_l2 
skopt_df['RMSPE'] = RMSPE 

In [ ]:
skopt_df.sort_values(by=['RMSPE'], ascending=True)